# 📓 Generative AI Module 3: High-Quality Neural Style Transfer (L-BFGS)
Welcome to Module 3! In this notebook, we implement **High-Quality Neural Style Transfer (Gatys et al., 2015)** matching the official PyTorch implementation.

### Key Improvements for High Aesthetics:
1. **L-BFGS Optimizer:** Second-order optimization for smooth, non-grainy pixel generation.
2. **Average Pooling Replacement:** Replaces VGG's `MaxPool2d` with `AvgPool2d` to prevent grid artifacts.
3. **In-Pipeline Loss Modules:** Embeds Content and Style loss evaluation directly inside the forward pass.


In [ ]:
import os
import urllib.request
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import clear_output
import ipywidgets as widgets
from ipywidgets import interact, FloatLogSlider, IntSlider

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Download Benchmark Images
We download real content and style benchmark images.

In [ ]:
content_url = "https://users.dcc.uchile.cl/~vbarrier/CC5219/data/content_img.jpg"
style_url = "https://users.dcc.uchile.cl/~vbarrier/CC5219/data/style_img.jpg"

if not os.path.exists("content_img.jpg"):
    !wget -q https://users.dcc.uchile.cl/~vbarrier/CC5219/data/content_img.jpg -O content_img.jpg
if not os.path.exists("style_img.jpg"):
    !wget -q https://users.dcc.uchile.cl/~vbarrier/CC5219/data/style_img.jpg -O style_img.jpg

if not os.path.exists("content_img.jpg"):
    urllib.request.urlretrieve(content_url, "content_img.jpg")
if not os.path.exists("style_img.jpg"):
    urllib.request.urlretrieve(style_url, "style_img.jpg")

In [ ]:
pil_content = Image.open("content_img.jpg").convert('RGB')
pil_style = Image.open("style_img.jpg").convert('RGB')

imsize = 512 if torch.cuda.is_available() else 128

loader = transforms.Compose([
    transforms.Resize((imsize, imsize)),
    transforms.ToTensor()
])
unloader = transforms.ToPILImage()

content_img = loader(pil_content).unsqueeze(0).to(device, torch.float)
style_img = loader(pil_style).unsqueeze(0).to(device, torch.float)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(unloader(content_img.cpu().squeeze(0)))
axes[0].set_title("Content Image")
axes[0].axis('off')
axes[1].imshow(unloader(style_img.cpu().squeeze(0)))
axes[1].set_title("Style Image")
axes[1].axis('off')
plt.show()

## 2. Custom Loss Modules & Gram Matrix
We define custom PyTorch modules that compute losses in-place during forward passes.

In [ ]:
class ContentLoss(nn.Module):
    def __init__(self, target,):
        super(ContentLoss, self).__init__()
        self.target = target.detach()

    def forward(self, input):
        self.loss = F.mse_loss(input, self.target)
        return input

def gram_matrix(input):
    a, b, c, d = input.size()
    features = input.view(a * b, c * d)
    G = torch.mm(features, features.t())
    return G.div(a * b * c * d)

class StyleLoss(nn.Module):
    def __init__(self, target_feature):
        super(StyleLoss, self).__init__()
        self.target = gram_matrix(target_feature).detach()

    def forward(self, input):
        G = gram_matrix(input)
        self.loss = F.mse_loss(G, self.target)
        return input

class Normalization(nn.Module):
    def __init__(self, mean, std):
        super(Normalization, self).__init__()
        self.mean = torch.tensor(mean).view(-1, 1, 1).to(device)
        self.std = torch.tensor(std).view(-1, 1, 1).to(device)

    def forward(self, img):
        return (img - self.mean) / self.std

## 3. Sequential Model Pipeline with AvgPool Replacement
We replace all `MaxPool2d` layers with `AvgPool2d` to ensure smooth gradient distribution across feature maps.

In [ ]:
cnn = models.vgg19(weights=models.VGG19_Weights.DEFAULT).features.to(device).eval()

cnn_normalization_mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
cnn_normalization_std = torch.tensor([0.229, 0.224, 0.225]).to(device)

content_layers_default = ['conv_4']
style_layers_default = ['conv_1', 'conv_2', 'conv_3', 'conv_4', 'conv_5']

def get_style_model_and_losses(cnn, normalization_mean, normalization_std,
                               style_img, content_img,
                               content_layers=content_layers_default,
                               style_layers=style_layers_default):
    normalization = Normalization(normalization_mean, normalization_std).to(device)
    content_losses = []
    style_losses = []
    model = nn.Sequential(normalization)

    i = 0
    for layer in cnn.children():
        if isinstance(layer, nn.Conv2d):
            i += 1
            name = f'conv_{i}'
        elif isinstance(layer, nn.ReLU):
            name = f'relu_{i}'
            layer = nn.ReLU(inplace=False)
        elif isinstance(layer, nn.MaxPool2d):
            name = f'pool_{i}'
            layer = nn.AvgPool2d(kernel_size=layer.kernel_size, stride=layer.stride)
        elif isinstance(layer, nn.BatchNorm2d):
            name = f'bn_{i}'
        else:
            raise RuntimeError(f'Unrecognized layer: {layer.__class__.__name__}')

        model.add_module(name, layer)

        if name in content_layers:
            target = model(content_img).detach()
            content_loss = ContentLoss(target)
            model.add_module(f"content_loss_{i}", content_loss)
            content_losses.append(content_loss)

        if name in style_layers:
            target_feature = model(style_img).detach()
            style_loss = StyleLoss(target_feature)
            model.add_module(f"style_loss_{i}", style_loss)
            style_losses.append(style_loss)

    for j in range(len(model) - 1, -1, -1):
        if isinstance(model[j], ContentLoss) or isinstance(model[j], StyleLoss):
            break
    model = model[:(j + 1)]

    return model, style_losses, content_losses

## 4. L-BFGS Optimization Loop
We optimize the input image directly using L-BFGS.

In [ ]:
def run_style_transfer(cnn, normalization_mean, normalization_std,
                       content_img, style_img, input_img, num_steps=300,
                       style_weight=1000000, content_weight=1):
    print('Building the style transfer model...')
    model, style_losses, content_losses = get_style_model_and_losses(
        cnn, normalization_mean, normalization_std, style_img, content_img)

    input_img.requires_grad_(True)
    model.eval()
    model.requires_grad_(False)

    optimizer = optim.LBFGS([input_img])

    print('Optimizing with L-BFGS...')
    run = [0]
    while run[0] <= num_steps:
        def closure():
            with torch.no_grad():
                input_img.clamp_(0, 1)

            optimizer.zero_grad()
            model(input_img)
            style_score = 0
            content_score = 0

            for sl in style_losses:
                style_score += sl.loss
            for cl in content_losses:
                content_score += cl.loss

            style_score *= style_weight
            content_score *= content_weight

            loss = style_score + content_score
            loss.backward()

            run[0] += 1
            if run[0] % 50 == 0 or run[0] == num_steps:
                clear_output(wait=True)
                fig, axes = plt.subplots(1, 3, figsize=(12, 4))
                axes[0].imshow(unloader(content_img.cpu().squeeze(0)))
                axes[0].set_title("Content")
                axes[0].axis('off')
                axes[1].imshow(unloader(input_img.data.clamp(0, 1).cpu().squeeze(0)))
                axes[1].set_title(f"L-BFGS Step {run[0]}/{num_steps}")
                axes[1].axis('off')
                axes[2].imshow(unloader(style_img.cpu().squeeze(0)))
                axes[2].set_title("Style")
                axes[2].axis('off')
                plt.tight_layout()
                plt.show()

            return style_score + content_score

        optimizer.step(closure)

    with torch.no_grad():
        input_img.clamp_(0, 1)

    return input_img

## 5. Execution & Interactive Controls
Run the optimization loop or adjust parameters interactively.

In [ ]:
input_img = content_img.clone()
output = run_style_transfer(cnn, cnn_normalization_mean, cnn_normalization_std,
                            content_img, style_img, input_img, num_steps=300,
                            style_weight=1000000, content_weight=1)

def interactive_lbfgs(style_w=1e6, content_w=1.0, steps=150):
    inp = content_img.clone()
    run_style_transfer(cnn, cnn_normalization_mean, cnn_normalization_std,
                       content_img, style_img, inp, num_steps=steps,
                       style_weight=style_w, content_weight=content_w)

interact(interactive_lbfgs,
         style_w=FloatLogSlider(value=1e6, base=10, min=3, max=8, step=0.5, description='Style Weight'),
         content_w=FloatLogSlider(value=1.0, base=10, min=-1, max=3, step=0.5, description='Content Weight'),
         steps=IntSlider(value=150, min=50, max=300, step=50, description='L-BFGS Steps'));